In [2]:
# ============================================================================
# CELL 1: Install Dependencies
# ============================================================================
print("📦 Installing required packages...")
!pip install -q langgraph groq arxiv PyPDF2 requests
print("✅ Packages installed!")

# ============================================================================
# CELL 2: Import Libraries and Setup
# ============================================================================
import os
import json
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
import arxiv
import requests
from pathlib import Path
import PyPDF2
from groq import Groq
from datetime import datetime
import time
import shutil
import zipfile

# Kaggle specific: Get API key from secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

try:
    GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
    print("✅ API key loaded from Kaggle secrets")
except:
    print("⚠️  GROQ_API_KEY not found in secrets!")
    print("   Get your FREE API key at: https://console.groq.com/keys")
    print("   Then add it in Kaggle:")
    print("   Settings > Add-ons > Secrets > Add Secret")
    print("   Name: GROQ_API_KEY")
    print("   Value: your-groq-api-key")
    GROQ_API_KEY = None

# Initialize Groq client
if GROQ_API_KEY:
    groq_client = Groq(api_key=GROQ_API_KEY)
    print("✅ Groq client initialized")
    print("   Model: llama-3.3-70b-versatile")
    print("   Free Tier: 30 requests/min, 14,400/day")
else:
    print("❌ Cannot proceed without API key")

print("\n" + "="*80)
print("🚀 MALAYALAM HTR RESEARCH AGENT - GROQ VERSION")
print("="*80 + "\n")

# ============================================================================
# CELL 3: State Definition
# ============================================================================
class ResearchState(TypedDict):
    topic: str
    output_folder: str
    papers_found: List[dict]
    downloaded_papers: List[dict]
    top_25_papers: List[dict]
    summaries: List[dict]
    comparison_report: str
    output_file: str

print("✅ State definition ready")

# ============================================================================
# CELL 4: Agent 1 - Multi-source Search
# ============================================================================
def search_papers_agent(state: ResearchState) -> ResearchState:
    """Search multiple academic databases for Malayalam HTR papers"""
    print("\n🔍 AGENT 1: Searching for Malayalam HTR papers...")
    print("-" * 80)
    
    all_papers = []
    
    # 1. ArXiv Search
    print("\n📚 Searching arXiv...")
    try:
        search_queries = [
            "Malayalam handwritten text recognition",
            "Malayalam HTR OCR",
            "Indic script handwriting recognition Malayalam"
        ]
        
        for query in search_queries:
            print(f"  Query: '{query}'")
            search = arxiv.Search(
                query=query,
                max_results=30,
                sort_by=arxiv.SortCriterion.Relevance
            )
            
            for result in search.results():
                if not any(p['title'] == result.title for p in all_papers):
                    all_papers.append({
                        'title': result.title,
                        'authors': [author.name for author in result.authors],
                        'year': result.published.year,
                        'abstract': result.summary,
                        'pdf_url': result.pdf_url,
                        'source': 'arXiv',
                        'entry_id': result.entry_id,
                        'is_free': True
                    })
            
            time.sleep(1)
        
        arxiv_count = len([p for p in all_papers if p['source'] == 'arXiv'])
        print(f"  ✓ Found {arxiv_count} unique papers from arXiv")
    except Exception as e:
        print(f"  ✗ ArXiv error: {e}")
    
    # 2. Semantic Scholar API
    print("\n📚 Searching Semantic Scholar...")
    try:
        ss_url = "https://api.semanticscholar.org/graph/v1/paper/search"
        
        search_terms = [
            'Malayalam handwritten text recognition',
            'Malayalam OCR deep learning',
            'Indic script recognition Malayalam'
        ]
        
        for term in search_terms:
            print(f"  Query: '{term}'")
            params = {
                'query': term,
                'limit': 30,
                'fields': 'title,authors,year,abstract,openAccessPdf,citationCount',
                'year': '2015-2025'
            }
            response = requests.get(ss_url, params=params, timeout=15)
            
            if response.status_code == 200:
                data = response.json()
                for paper in data.get('data', []):
                    if paper.get('openAccessPdf'):
                        if not any(p['title'] == paper.get('title') for p in all_papers):
                            all_papers.append({
                                'title': paper.get('title', 'Unknown'),
                                'authors': [a.get('name', 'Unknown') for a in paper.get('authors', [])[:5]],
                                'year': paper.get('year', 'Unknown'),
                                'abstract': paper.get('abstract', 'No abstract available'),
                                'pdf_url': paper['openAccessPdf'].get('url'),
                                'source': 'Semantic Scholar',
                                'citation_count': paper.get('citationCount', 0),
                                'is_free': True
                            })
            
            time.sleep(1)
        
        ss_count = len([p for p in all_papers if p['source'] == 'Semantic Scholar'])
        print(f"  ✓ Found {ss_count} unique papers from Semantic Scholar")
    except Exception as e:
        print(f"  ✗ Semantic Scholar error: {e}")
    
    print(f"\n✅ TOTAL FOUND: {len(all_papers)} unique free papers")
    
    state['papers_found'] = all_papers
    return state

print("✅ Search agent ready")

# ============================================================================
# CELL 5: Agent 2 - Download Papers
# ============================================================================
def download_papers_agent(state: ResearchState) -> ResearchState:
    """Download free papers to temporary folder"""
    print("\n📥 AGENT 2: Downloading papers to temporary folder...")
    print("-" * 80)
    
    # Create temporary folder
    temp_path = "/kaggle/working/temp_all_papers"
    Path(temp_path).mkdir(parents=True, exist_ok=True)
    print(f"📁 Temporary folder: {temp_path}")
    
    downloaded = []
    failed = []
    
    total = len(state['papers_found'])
    print(f"\n📊 Processing {total} papers...\n")
    
    for i, paper in enumerate(state['papers_found'], 1):
        if not paper.get('is_free') or not paper.get('pdf_url'):
            continue
        
        if i % 5 == 0:
            print(f"  Progress: {i}/{total} papers processed...")
        
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
            response = requests.get(paper['pdf_url'], timeout=30, headers=headers)
            
            if response.status_code == 200 and len(response.content) > 1000:
                safe_title = "".join(c for c in paper['title'][:40] if c.isalnum() or c in (' ', '-', '_')).strip()
                safe_title = safe_title.replace(' ', '_')
                filename = f"{safe_title}_{paper['year']}.pdf"
                filepath = os.path.join(temp_path, filename)
                
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                
                paper['local_path'] = filepath
                paper['filename'] = filename
                downloaded.append(paper)
            else:
                failed.append(paper['title'])
        
        except Exception as e:
            failed.append(paper['title'])
            continue
        
        time.sleep(0.5)
    
    print(f"\n✅ Successfully downloaded: {len(downloaded)} papers to temp folder")
    if failed:
        print(f"⚠️  Failed: {len(failed)} papers")
    
    state['downloaded_papers'] = downloaded
    return state

print("✅ Download agent ready")

# ============================================================================
# CELL 6: Agent 3 - Ranking Papers (GROQ)
# ============================================================================
def ranking_agent(state: ResearchState) -> ResearchState:
    """Use Groq (Llama 3.3) to rank papers by effectiveness"""
    print("\n🎯 AGENT 3: Ranking papers by effectiveness...")
    print("-" * 80)
    
    if len(state['downloaded_papers']) == 0:
        print("⚠️  No papers to rank!")
        state['top_25_papers'] = []
        return state
    
    print(f"📊 Analyzing {len(state['downloaded_papers'])} papers...")
    
    papers_text = ""
    for i, paper in enumerate(state['downloaded_papers'], 1):
        papers_text += f"\n{i}. {paper['title']}\n"
        papers_text += f"   Year: {paper['year']} | Source: {paper['source']}\n"
        if 'citation_count' in paper:
            papers_text += f"   Citations: {paper['citation_count']}\n"
        abstract = paper.get('abstract') or 'No abstract available'
        papers_text += f"   Abstract: {abstract[:350]}...\n"
    
    prompt = f"""You are an expert in Handwritten Text Recognition (HTR) for Indic scripts, especially Malayalam.

Analyze these {len(state['downloaded_papers'])} papers about Malayalam HTR pipelines and rank the TOP 25 (or all if fewer).

RANKING CRITERIA:
1. Pipeline Architecture Innovation (CNN-LSTM, Attention, Transformers)
2. Performance Metrics (CER/WER improvements)
3. Recency (2020-2025 papers preferred)
4. Dataset Quality & Real-world Applicability
5. Technical Depth & Reproducibility

Papers:
{papers_text}

Return ONLY valid JSON in this EXACT format:
{{
  "rankings": [
    {{"index": 1, "title": "paper title", "score": 45, "reason": "2-3 sentences why"}},
    ...
  ]
}}

Include TOP 25 papers. Score out of 50. Be specific about pipelines."""
    
    try:
        print("  🤖 Calling Groq (Llama 3.3-70B) for ranking analysis...")
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=6000,
            temperature=0.3
        )
        
        content = response.choices[0].message.content
        json_start = content.find('{')
        json_end = content.rfind('}') + 1
        
        if json_start != -1 and json_end > json_start:
            json_str = content[json_start:json_end]
            rankings_data = json.loads(json_str)
            
            top_25 = []
            for rank_info in rankings_data['rankings'][:25]:
                idx = rank_info['index'] - 1
                if 0 <= idx < len(state['downloaded_papers']):
                    paper = state['downloaded_papers'][idx].copy()
                    paper['rank'] = len(top_25) + 1
                    paper['ranking_reason'] = rank_info['reason']
                    paper['effectiveness_score'] = rank_info.get('score', 0)
                    top_25.append(paper)
            
            print(f"✅ Ranked top {len(top_25)} papers")
            state['top_25_papers'] = top_25
        else:
            raise ValueError("Could not parse JSON")
        
    except Exception as e:
        print(f"⚠️  Ranking error: {e}")
        print("   Using fallback (citations + year)...")
        
        sorted_papers = sorted(
            state['downloaded_papers'],
            key=lambda x: (x.get('citation_count', 0), int(x['year']) if str(x['year']).isdigit() else 0),
            reverse=True
        )[:25]
        
        for i, paper in enumerate(sorted_papers, 1):
            paper['rank'] = i
            paper['ranking_reason'] = f"Citations: {paper.get('citation_count', 0)}, Year: {paper['year']}"
            paper['effectiveness_score'] = 0
        
        state['top_25_papers'] = sorted_papers
    
    return state

print("✅ Ranking agent ready")

# ============================================================================
# CELL 7: Agent 4 - Summarization (GROQ)
# ============================================================================
def summarization_agent(state: ResearchState) -> ResearchState:
    """Generate detailed summaries of top papers using Groq"""
    print("\n📝 AGENT 4: Generating detailed summaries...")
    print("-" * 80)
    
    summaries = []
    total = len(state['top_25_papers'])
    
    for paper in state['top_25_papers']:
        print(f"\n  [{paper['rank']}/{total}] {paper['title'][:50]}...")
        
        try:
            pdf_text = ""
            with open(paper['local_path'], 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                num_pages = min(12, len(pdf_reader.pages))
                for page_num in range(num_pages):
                    try:
                        pdf_text += pdf_reader.pages[page_num].extract_text() + "\n"
                    except:
                        continue
            
            if not pdf_text.strip():
                print(f"    ⚠️  Could not extract text")
                continue
            
            prompt = f"""Analyze this Malayalam HTR paper:

Title: {paper['title']}
Year: {paper['year']}
Abstract: {paper.get('abstract', 'No abstract')}

Text (first 12 pages):
{pdf_text[:10000]}

Provide structured analysis:

**1. MAIN CONTRIBUTION**
Key innovation (2-3 sentences)

**2. PIPELINE ARCHITECTURE**
- Type (Traditional/CNN-LSTM/Attention/Transformer)
- Specific models/layers
- Preprocessing & decoding

**3. DATASET**
- Name, size, type
- Splits, augmentation

**4. RESULTS**
- CER/WER/Accuracy metrics
- Comparison with baselines

**5. APPLICABLE TECHNIQUES**
5-7 specific techniques for HTR projects

**6. LIMITATIONS**
Gaps or limitations

Be technical and concise."""
            
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                temperature=0.3
            )
            
            summary = {
                'rank': paper['rank'],
                'title': paper['title'],
                'authors': paper['authors'],
                'year': paper['year'],
                'source': paper['source'],
                'pdf_url': paper['pdf_url'],
                'filename': paper['filename'],
                'ranking_reason': paper['ranking_reason'],
                'effectiveness_score': paper.get('effectiveness_score', 0),
                'detailed_summary': response.choices[0].message.content,
                'pdf_path': paper['local_path']
            }
            
            summaries.append(summary)
            print(f"    ✓ Complete")
            
            # Small delay to respect rate limits
            time.sleep(1)
            
        except Exception as e:
            print(f"    ✗ Error: {str(e)[:50]}")
            continue
    
    print(f"\n✅ Generated {len(summaries)} summaries")
    
    state['summaries'] = summaries
    return state

print("✅ Summarization agent ready")

# ============================================================================
# CELL 8: Agent 5 - Comparative Analysis (GROQ)
# ============================================================================
def comparison_agent(state: ResearchState) -> ResearchState:
    """Generate comparative analysis using Groq"""
    print("\n🔬 AGENT 5: Generating comparative analysis...")
    print("-" * 80)
    
    summaries_text = ""
    for summary in state['summaries']:
        summaries_text += f"\n{'='*80}\n"
        summaries_text += f"PAPER #{summary['rank']}: {summary['title']}\n"
        summaries_text += f"Year: {summary['year']} | Score: {summary['effectiveness_score']}/50\n"
        summaries_text += f"{summary['detailed_summary']}\n"
    
    prompt = f"""Analyze Malayalam HTR pipeline evolution across {len(state['summaries'])} papers (2015-2025).

Papers:
{summaries_text}

Create comprehensive WRITTEN comparative analysis (NO graphs, NO tables, only detailed prose):

# MALAYALAM HTR COMPARATIVE ANALYSIS

## 1. WHY PAPER #1 IS MOST EFFECTIVE
Write 4-5 detailed paragraphs explaining:
- What makes Paper #1 the most effective compared to all other papers
- Specific architectural advantages over Papers #2, #3, #4, #5 (name them and explain differences)
- Performance metrics comparison (exact CER/WER numbers) vs other top papers
- Why its pipeline/approach is superior for Malayalam HTR (be specific about techniques)
- What innovations it brings that others don't have

## 2. DETAILED RANKING JUSTIFICATION (TOP 10)
For EACH of the top 10 papers, write 2-3 paragraphs:

**Paper #2:**
- Why it ranks #2 specifically (what does it do better than #3, #4, #5?)
- Direct comparisons with lower-ranked papers (cite specific papers and metrics)
- What it does well and where it falls short of #1

**Paper #3:**
- Why it beats #4, #5, #6... (specific architectural/performance reasons)
- Comparison with Papers #1 and #2
- Its unique strengths

Continue this pattern for Papers #4, #5, #6, #7, #8, #9, #10

## 3. PIPELINE EVOLUTION NARRATIVE
Write 5-6 paragraphs describing how Malayalam HTR evolved from 2015-2025:
- Early approaches (2015-2017): Which papers used what methods? Why were they limited?
- Deep learning era (2018-2020): Which papers introduced CNNs/LSTMs? What improved?
- Attention mechanisms (2021-2023): Which papers pioneered attention for Malayalam? Impact?
- State-of-the-art (2024-2025): Current best practices from top-ranked papers

## 4. ARCHITECTURE COMPARISON IN PROSE
Write detailed paragraphs comparing architectural approaches:
- Traditional methods (which papers, limitations)
- Pure CNNs (which papers used them, performance)
- CNN-LSTM hybrids (most effective combinations from which papers)
- Attention mechanisms (which papers, CER/WER improvements)
- Transformers (cutting-edge papers, results)

For each, cite specific paper ranks and explain why some architectures worked better.

## 5. PERFORMANCE ANALYSIS
Write 3-4 paragraphs:
- Best CER achieved (which paper, exact number, why it's best)
- Best WER achieved (which paper, comparison with others)
- Best accuracy (which paper, methodology)
- Why these papers achieved superior performance vs others

## 6. MOST EFFECTIVE TECHNIQUES EXPLAINED
Write detailed explanations (not a list) of the 10 most impactful techniques:
- Technique 1: [name] - Which papers used it (#1, #5, #12)? How did it improve results? Why is it effective for Malayalam specifically?
- Technique 2: [Continue same detailed format]
...continue for all 10

## 7. DATASET AND TRAINING INSIGHTS
Write 4-5 paragraphs about:
- Evolution of datasets (early vs modern, which papers used what)
- Most effective augmentation strategies (from which papers)
- Training approaches that worked best (papers and techniques)

## 8. GAPS AND FUTURE DIRECTIONS
Write 3-4 paragraphs:
- What problems remain unsolved across all papers
- Promising but underexplored approaches (cite papers)
- Recommendations for future research

## 9. ACTIONABLE RECOMMENDATIONS FOR 2025
Write 10-15 paragraphs with concrete recommendations:
- Best architecture to use (based on Paper #X, because...)
- Dataset strategies (Papers #Y and #Z showed that...)
- Training techniques (Paper #A's approach of... outperformed others)
- Preprocessing steps (Papers #B, #C, #D all used... with success)
Continue with specific, detailed recommendations citing paper ranks

IMPORTANT: 
- Write in flowing paragraphs, NOT bullet points
- Always cite specific paper ranks when making comparisons
- Include exact metrics (CER/WER/Accuracy numbers)
- Explain WHY things work, not just WHAT works
- Make direct comparisons between papers constantly"""
    
    try:
        print("  🤖 Groq analyzing patterns...")
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=8000,
            temperature=0.3
        )
        
        state['comparison_report'] = response.choices[0].message.content
        print("✅ Analysis complete")
        
    except Exception as e:
        print(f"⚠️  Error: {e}")
        state['comparison_report'] = f"Error generating comparison: {e}"
    
    return state

print("✅ Comparison agent ready")

# ============================================================================
# CELL 9: Agent 6 - Export to Files
# ============================================================================
def export_to_file_agent(state: ResearchState) -> ResearchState:
    """Create comprehensive report files AND copy top 25 papers to final folder"""
    print("\n📄 AGENT 6: Creating final reports and organizing top 25 papers...")
    print("-" * 80)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_base = f"/kaggle/working/{state['output_folder']}"
    Path(output_base).mkdir(parents=True, exist_ok=True)
    
    # COPY ONLY TOP 25 PAPERS TO FINAL FOLDER
    print(f"\n📁 Copying top 25 ranked papers to: {output_base}/")
    copied_count = 0
    for summary in state['summaries']:
        try:
            src = summary['pdf_path']
            rank_prefix = f"Rank_{summary['rank']:02d}_"
            dst_filename = rank_prefix + summary['filename']
            dst = os.path.join(output_base, dst_filename)
            
            shutil.copy2(src, dst)
            copied_count += 1
            
            if copied_count <= 5 or copied_count == len(state['summaries']):
                print(f"  ✓ Copied Paper #{summary['rank']}: {summary['filename'][:50]}")
        except Exception as e:
            print(f"  ✗ Failed to copy Paper #{summary['rank']}: {e}")
    
    print(f"\n✅ Copied {copied_count} top-ranked papers to final folder")
    
    # Create ZIP file of top 25 papers
    print(f"\n📦 Creating ZIP file of top 25 papers...")
    try:
        zip_path = os.path.join(output_base, f"top_25_papers_{timestamp}.zip")
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for summary in state['summaries']:
                rank_prefix = f"Rank_{summary['rank']:02d}_"
                dst_filename = rank_prefix + summary['filename']
                file_path = os.path.join(output_base, dst_filename)
                if os.path.exists(file_path):
                    zipf.write(file_path, dst_filename)
        print(f"✅ ZIP created: {zip_path}")
        print(f"   Download this single file to get all papers!")
    except Exception as e:
        print(f"⚠️  Could not create ZIP: {e}")
    
    # Create reports
    output_file = os.path.join(output_base, f"malayalam_htr_analysis_{timestamp}.md")
    
    try:
        # Markdown version
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("# MALAYALAM HTR RESEARCH ANALYSIS\n")
            f.write(f"## Top {len(state['summaries'])} Papers - 2015-2025\n\n")
            f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"**Powered by:** Groq (Llama 3.3-70B)\n\n")
            
            f.write("---\n\n")
            f.write("## COMPARATIVE ANALYSIS\n\n")
            f.write(state['comparison_report'])
            f.write("\n\n---\n\n")
            
            f.write("## QUICK REFERENCE\n\n")
            f.write("| Rank | Title | Year | Score |\n")
            f.write("|------|-------|------|-------|\n")
            for s in state['summaries']:
                title_short = s['title'][:50] + "..." if len(s['title']) > 50 else s['title']
                f.write(f"| #{s['rank']} | {title_short} | {s['year']} | {s['effectiveness_score']}/50 |\n")
            
            f.write("\n\n---\n\n")
            f.write("## DETAILED SUMMARIES\n\n")
            
            for s in state['summaries']:
                f.write(f"\n### PAPER #{s['rank']}: {s['title']}\n\n")
                f.write(f"**Authors:** {', '.join(s['authors'])}\n\n")
                f.write(f"**Year:** {s['year']} | **Score:** {s['effectiveness_score']}/50\n\n")
                f.write(f"**Why Ranked #{s['rank']}:** {s['ranking_reason']}\n\n")
                f.write(f"**File:** `Rank_{s['rank']:02d}_{s['filename']}`\n\n")
                f.write("---\n\n")
                f.write(s['detailed_summary'])
                f.write("\n\n---\n\n")
        
        print(f"✅ Markdown: {output_file}")
        
        # Plain text version
        txt_file = output_file.replace('.md', '.txt')
        with open(txt_file, 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n")
            f.write("MALAYALAM HTR RESEARCH ANALYSIS\n")
            f.write(f"Top {len(state['summaries'])} Papers (2015-2025)\n")
            f.write("Powered by: Groq (Llama 3.3-70B)\n")
            f.write("="*80 + "\n\n")
            
            f.write("COMPARATIVE ANALYSIS\n" + "-"*80 + "\n\n")
            f.write(state['comparison_report'])
            f.write("\n\n" + "="*80 + "\n\n")
            
            for s in state['summaries']:
                f.write(f"PAPER #{s['rank']}: {s['title']}\n")
                f.write("-"*80 + "\n")
                f.write(f"Authors: {', '.join(s['authors'])}\n")
                f.write(f"Year: {s['year']} | Score: {s['effectiveness_score']}/50\n")
                f.write(f"File: Rank_{s['rank']:02d}_{s['filename']}\n\n")
                f.write(f"Why #{s['rank']}: {s['ranking_reason']}\n\n")
                f.write(s['detailed_summary'])
                f.write("\n\n" + "="*80 + "\n\n")
        
        print(f"✅ Plain text: {txt_file}")
        
        state['output_file'] = output_file
        
    except Exception as e:
        print(f"⚠️  Error: {e}")
        state['output_file'] = f"Error: {e}"
    
    return state

print("✅ Export agent ready")

# ============================================================================
# CELL 10: Build Workflow
# ============================================================================
def build_workflow():
    """Build LangGraph workflow"""
    workflow = StateGraph(ResearchState)
    
    workflow.add_node("search", search_papers_agent)
    workflow.add_node("download", download_papers_agent)
    workflow.add_node("rank", ranking_agent)
    workflow.add_node("summarize", summarization_agent)
    workflow.add_node("compare", comparison_agent)
    workflow.add_node("export", export_to_file_agent)
    
    workflow.set_entry_point("search")
    workflow.add_edge("search", "download")
    workflow.add_edge("download", "rank")
    workflow.add_edge("rank", "summarize")
    workflow.add_edge("summarize", "compare")
    workflow.add_edge("compare", "export")
    workflow.add_edge("export", END)
    
    return workflow.compile()

print("✅ Workflow builder ready")

# ============================================================================
# CELL 11: Main Execution Function
# ============================================================================
def run_research_pipeline(output_folder: str = "malayalam_htr_papers"):
    """Run the complete pipeline"""
    
    print("\n" + "="*80)
    print("🚀 STARTING MALAYALAM HTR RESEARCH PIPELINE")
    print("="*80 + "\n")
    
    initial_state = {
        'topic': 'Malayalam HTR handwritten text recognition pipelines',
        'output_folder': output_folder,
        'papers_found': [],
        'downloaded_papers': [],
        'top_25_papers': [],
        'summaries': [],
        'comparison_report': '',
        'output_file': ''
    }
    
    app = build_workflow()
    
    print("⏱️  Estimated time: 15-30 minutes\n")
    start_time = time.time()
    
    final_state = app.invoke(initial_state)
    
    elapsed = time.time() - start
    # ============================================================================
# CELL 11: Main Execution Function (COMPLETE)
# ============================================================================
def run_research_pipeline(output_folder: str = "malayalam_htr_papers"):
    """Run the complete pipeline"""
    
    print("\n" + "="*80)
    print("🚀 STARTING MALAYALAM HTR RESEARCH PIPELINE")
    print("="*80 + "\n")
    
    initial_state = {
        'topic': 'Malayalam HTR handwritten text recognition pipelines',
        'output_folder': output_folder,
        'papers_found': [],
        'downloaded_papers': [],
        'top_25_papers': [],
        'summaries': [],
        'comparison_report': '',
        'output_file': ''
    }
    
    app = build_workflow()
    
    print("⏱️  Estimated time: 15-30 minutes\n")
    start_time = time.time()
    
    final_state = app.invoke(initial_state)
    
    elapsed = time.time() - start_time
    
    print("\n" + "="*80)
    print("✅ PIPELINE COMPLETE!")
    print("="*80)
    print(f"\n⏱️  Time taken: {elapsed/60:.1f} minutes")
    print(f"\n📊 RESULTS:")
    print(f"  • Papers found: {len(final_state['papers_found'])}")
    print(f"  • Downloaded: {len(final_state['downloaded_papers'])}")
    print(f"  • Analyzed: {len(final_state['top_25_papers'])}")
    print(f"\n📁 OUTPUT LOCATION:")
    print(f"  • Folder: /kaggle/working/{output_folder}/")
    print(f"  • Report: {os.path.basename(final_state.get('output_file', 'N/A'))}")
    print(f"\n💡 Download files from Kaggle Output panel (right sidebar) ➡️")
    print("="*80 + "\n")
    
    return final_state

print("✅ Main execution function ready")
print("\n" + "="*80)
print("✅ ALL AGENTS LOADED - READY TO RUN!")
print("="*80)

# ============================================================================
# CELL 12: RUN THE PIPELINE
# ============================================================================
print("\n🎬 EXECUTING PIPELINE...\n")

result = run_research_pipeline(output_folder="malayalam_htr_papers_2025")

print("\n📥 TO DOWNLOAD YOUR RESULTS:")
print("1. Look at the right sidebar 'Output' panel in Kaggle")
print("2. Download the ZIP file: top_25_papers_[timestamp].zip")
print("3. Or download the entire folder: malayalam_htr_papers_2025/")
print("4. Open the .txt or .md file to read the analysis!")
print("\n✨ Done! 🎉")

📦 Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00
✅ Packages installed!
✅ API key loaded from Kaggle secrets
✅ Groq client initialized
   Model: llama-3.3-70b-versatile
   Free Tier: 30 requests/min, 14,400/day

🚀 MALAYALAM HTR RESEARCH AGENT - GROQ VERSION

✅ State definition ready
✅ Search agent ready
✅ Download agent ready
✅ Ranking agent ready
✅ Summarization agent ready
✅ Comparison agent ready
✅ Export agent ready
✅ Workflow builder ready
✅ Main execution function ready

✅ ALL AGENTS LOADED - READY TO RUN!

🎬 EXECUTING PIPELINE...


🚀 STARTING MALAYALAM HTR RESEARCH PIPELINE

⏱️  Estimated time: 15-30 minutes


🔍 AGENT 1: Searching for Malayalam HTR papers...
--------------------------------------------------------------------------------

📚 Searching arXiv...
  Query: 'Malayalam handwritten text recognition'


/tmp/ipykernel_47/3526132441.py:96: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


  Query: 'Malayalam HTR OCR'
  Query: 'Indic script handwriting recognition Malayalam'
  ✓ Found 81 unique papers from arXiv

📚 Searching Semantic Scholar...
  Query: 'Malayalam handwritten text recognition'
  Query: 'Malayalam OCR deep learning'
  Query: 'Indic script recognition Malayalam'
  ✓ Found 60 unique papers from Semantic Scholar

✅ TOTAL FOUND: 141 unique free papers

📥 AGENT 2: Downloading papers to temporary folder...
--------------------------------------------------------------------------------
📁 Temporary folder: /kaggle/working/temp_all_papers

📊 Processing 141 papers...

  Progress: 5/141 papers processed...
  Progress: 10/141 papers processed...
  Progress: 15/141 papers processed...
  Progress: 20/141 papers processed...
  Progress: 25/141 papers processed...
  Progress: 30/141 papers processed...
  Progress: 35/141 papers processed...
  Progress: 40/141 papers processed...
  Progress: 45/141 papers processed...
  Progress: 50/141 papers processed...
  Progress: 55

KeyboardInterrupt: 

In [10]:
# ============================================================================
# CELL 1: Install Dependencies
# ============================================================================
print("📦 Installing required packages...")
!pip install -q groq PyPDF2
print("✅ Packages installed!")

# ============================================================================
# CELL 2: Setup and Extract ZIP
# ============================================================================
import os
import json
import zipfile
from pathlib import Path
import PyPDF2
from groq import Groq
from datetime import datetime
import time

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ Groq client ready")

# Extract ZIP file
print("\n📦 Extracting papers from ZIP...")
zip_path = "/kaggle/input/top-ranked-papers-25"
extract_to = "/kaggle/working/extracted_papers"
Path(extract_to).mkdir(parents=True, exist_ok=True)

# Find and extract all ZIP files in the input folder
zip_files = list(Path(zip_path).glob("*.zip"))
print(f"Found {len(zip_files)} ZIP file(s)")

for zip_file in zip_files:
    print(f"Extracting: {zip_file.name}")
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

pdf_files = list(Path(extract_to).glob("*.pdf"))
print(f"✅ Extracted {len(pdf_files)} PDF files")

# ============================================================================
# CELL 3: Analyze Papers
# ============================================================================

def analyze_papers(papers_folder: str):
    """Analyze papers and generate summaries + comparison"""
    
    print("\n" + "="*80)
    print("📚 ANALYZING YOUR 25 RANKED PAPERS")
    print("="*80 + "\n")
    
    # Get all PDFs
    pdf_files = sorted(list(Path(papers_folder).glob("*.pdf")))
    
    if not pdf_files:
        print(f"❌ No PDF files found in {papers_folder}")
        return
    
    print(f"✅ Found {len(pdf_files)} PDF files\n")
    
    summaries = []
    
    # ============================================================================
    # STEP 1: Generate Summaries
    # ============================================================================
    print("📝 GENERATING SUMMARIES...")
    print("-" * 80)
    
    for i, pdf_path in enumerate(pdf_files, 1):
        print(f"\n[{i}/{len(pdf_files)}] {pdf_path.name[:70]}...")
        
        try:
            # Extract text from PDF
            pdf_text = ""
            with open(pdf_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                num_pages = min(15, len(pdf_reader.pages))
                for page_num in range(num_pages):
                    try:
                        pdf_text += pdf_reader.pages[page_num].extract_text() + "\n"
                    except:
                        continue
            
            if not pdf_text.strip():
                print(f"  ⚠️  Could not extract text, skipping...")
                continue
            
            # Get title from filename
            title = pdf_path.stem.replace('_', ' ')
            
            # Analyze with Groq
            prompt = f"""Analyze this Malayalam HTR research paper:

Title: {title}

Paper Text (first 15 pages):
{pdf_text[:12000]}

Provide structured analysis:

**1. MAIN CONTRIBUTION**
Key innovation (2-3 sentences)

**2. PIPELINE ARCHITECTURE**
- Type (Traditional/CNN-LSTM/Attention/Transformer)
- Specific models/layers used
- Preprocessing & decoding methods

**3. DATASET**
- Name, size, type of dataset
- Train/test splits
- Data augmentation techniques

**4. RESULTS**
- CER/WER/Accuracy metrics achieved
- Comparison with baseline methods

**5. APPLICABLE TECHNIQUES**
List 5-7 specific techniques that can be applied to HTR projects

**6. LIMITATIONS**
Gaps or limitations in the approach

Be technical and concise."""
            
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                temperature=0.3
            )
            
            summary = {
                'rank': i,
                'title': title,
                'filename': pdf_path.name,
                'detailed_summary': response.choices[0].message.content,
                'pdf_path': str(pdf_path)
            }
            
            summaries.append(summary)
            print(f"  ✓ Complete")
            
            # Respect rate limits
            time.sleep(2)
            
        except Exception as e:
            print(f"  ✗ Error: {str(e)[:100]}")
            continue
    
    print(f"\n✅ Generated {len(summaries)} summaries")
    
    if len(summaries) == 0:
        print("❌ No summaries generated. Cannot proceed with comparison.")
        return None
    
    # ============================================================================
    # STEP 2: Generate Comparative Analysis
    # ============================================================================
    print("\n🔬 GENERATING COMPARATIVE ANALYSIS...")
    print("-" * 80)
    
    summaries_text = ""
    for summary in summaries:
        summaries_text += f"\n{'='*80}\n"
        summaries_text += f"PAPER #{summary['rank']}: {summary['title']}\n"
        summaries_text += f"{summary['detailed_summary']}\n"
    
    prompt = f"""Analyze Malayalam HTR pipeline evolution across {len(summaries)} papers.

Papers:
{summaries_text}

Create comprehensive comparative analysis in detailed prose:

# MALAYALAM HTR COMPARATIVE ANALYSIS

## 1. OVERVIEW
Write 3-4 paragraphs summarizing the overall landscape of Malayalam HTR research based on these papers. Discuss the progression of techniques and current state.

## 2. PIPELINE EVOLUTION
Write 5-6 paragraphs describing:
- Traditional approaches and their limitations
- Deep learning methods (CNNs, LSTMs)
- Attention mechanisms and transformers
- Current state-of-the-art techniques
Cite specific papers by rank number.

## 3. KEY ARCHITECTURAL PATTERNS
Write 4-5 paragraphs comparing:
- Feature extraction methods across papers
- Sequence modeling approaches (RNN vs CNN vs Transformer)
- Decoding strategies (CTC, attention-based, etc.)
- What works best for Malayalam specifically and why

## 4. PERFORMANCE TRENDS
Write 3-4 paragraphs about:
- Best CER/WER/Accuracy achieved (cite paper rank)
- Performance improvements over time
- Which techniques give best results
- Include specific numbers from papers

## 5. DATASETS AND TRAINING
Write 3-4 paragraphs on:
- Common datasets used across papers
- Data augmentation strategies that work
- Training best practices and hyperparameters

## 6. MOST EFFECTIVE TECHNIQUES
Write detailed paragraphs explaining the 10 most impactful techniques for Malayalam HTR. For each technique, mention which papers (by rank) used it and why it's effective.

## 7. GAPS AND FUTURE WORK
Write 3-4 paragraphs on:
- Unsolved problems in Malayalam HTR
- Promising research directions
- Practical recommendations for practitioners

## 8. ACTIONABLE RECOMMENDATIONS
Write 5-6 paragraphs with concrete recommendations:
- Best architecture to use based on findings
- Dataset preparation strategies
- Training approaches that work
- Common pitfalls to avoid

IMPORTANT: 
- Write in flowing paragraphs, NOT bullet points
- Always cite specific paper ranks when making claims
- Include exact metrics (CER/WER/Accuracy) when mentioned
- Explain WHY techniques work, not just WHAT they are"""
    
    try:
        print("  🤖 Groq analyzing patterns (this may take 1-2 minutes)...")
        
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=8000,
            temperature=0.3
        )
        
        comparison_report = response.choices[0].message.content
        print("✅ Comparative analysis complete")
        
    except Exception as e:
        print(f"⚠️  Error: {e}")
        comparison_report = f"Error generating comparison: {e}"
    
    # ============================================================================
    # STEP 3: Export Results
    # ============================================================================
    print("\n📄 EXPORTING RESULTS...")
    print("-" * 80)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_folder = "/kaggle/working/analysis_results"
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # Markdown Report
    md_file = os.path.join(output_folder, f"malayalam_htr_analysis_{timestamp}.md")
    with open(md_file, 'w', encoding='utf-8') as f:
        f.write("# MALAYALAM HTR RESEARCH ANALYSIS\n\n")
        f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"**Papers Analyzed:** {len(summaries)}\n")
        f.write(f"**Powered by:** Groq (Llama 3.3-70B)\n\n")
        
        f.write("---\n\n")
        f.write("## COMPARATIVE ANALYSIS\n\n")
        f.write(comparison_report)
        f.write("\n\n---\n\n")
        
        f.write("## QUICK REFERENCE\n\n")
        f.write("| Rank | Title |\n")
        f.write("|------|-------|\n")
        for s in summaries:
            title_short = s['title'][:80] + "..." if len(s['title']) > 80 else s['title']
            f.write(f"| #{s['rank']} | {title_short} |\n")
        
        f.write("\n\n---\n\n")
        f.write("## DETAILED SUMMARIES\n\n")
        
        for s in summaries:
            f.write(f"\n### PAPER #{s['rank']}: {s['title']}\n\n")
            f.write(f"**File:** `{s['filename']}`\n\n")
            f.write("---\n\n")
            f.write(s['detailed_summary'])
            f.write("\n\n---\n\n")
    
    # Text Report
    txt_file = md_file.replace('.md', '.txt')
    with open(txt_file, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("MALAYALAM HTR RESEARCH ANALYSIS\n")
        f.write(f"Papers Analyzed: {len(summaries)}\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        
        f.write("COMPARATIVE ANALYSIS\n" + "-"*80 + "\n\n")
        f.write(comparison_report)
        f.write("\n\n" + "="*80 + "\n\n")
        
        f.write("DETAILED SUMMARIES\n" + "="*80 + "\n\n")
        for s in summaries:
            f.write(f"PAPER #{s['rank']}: {s['title']}\n")
            f.write("-"*80 + "\n")
            f.write(f"File: {s['filename']}\n\n")
            f.write(s['detailed_summary'])
            f.write("\n\n" + "="*80 + "\n\n")
    
    # JSON Export
    json_file = os.path.join(output_folder, f"summaries_{timestamp}.json")
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump({
            'summaries': summaries,
            'comparison': comparison_report,
            'timestamp': timestamp,
            'total_papers': len(summaries)
        }, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Results exported to:")
    print(f"  📄 Markdown: {md_file}")
    print(f"  📄 Text: {txt_file}")
    print(f"  📄 JSON: {json_file}")
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    print(f"\n📊 SUMMARY:")
    print(f"  • Papers analyzed: {len(summaries)}")
    print(f"  • Output folder: {output_folder}")
    print("\n💡 Download files from Kaggle Output panel (right sidebar) ➡️")
    print("="*80 + "\n")
    
    return {
        'summaries': summaries,
        'comparison_report': comparison_report,
        'output_files': {
            'markdown': md_file,
            'text': txt_file,
            'json': json_file
        }
    }

# ============================================================================
# CELL 4: RUN ANALYSIS (CORRECTED)
# ============================================================================

print("\n🚀 Starting analysis of your 25 ranked papers...")

# Point directly to your dataset (no extraction needed!)
result = analyze_papers("/kaggle/input/top-ranked-papers-25")

print("\n✨ Done! Check the 'analysis_results' folder in Output panel!")


📦 Installing required packages...
✅ Packages installed!
✅ Groq client ready

📦 Extracting papers from ZIP...
Found 0 ZIP file(s)
✅ Extracted 0 PDF files

🚀 Starting analysis of your 25 ranked papers...

📚 ANALYZING YOUR 25 RANKED PAPERS

✅ Found 25 PDF files

📝 GENERATING SUMMARIES...
--------------------------------------------------------------------------------

[1/25] Rank_01_TMIXT_A_process_flow_for_Transcribing_M_2019.pdf...
  ✓ Complete

[2/25] Rank_02_Handwritten_Character_Recognition_In_Mal_2014.pdf...
  ✓ Complete

[3/25] Rank_03_Offline_Handwritten_Recognition_of_Malay_2017.pdf...
  ✓ Complete

[4/25] Rank_04_Spatial_Domain_Feature_Extraction_Method_2021.pdf...
  ✓ Complete

[5/25] Rank_05_HMM-based_Indic_Handwritten_Word_Recogni_2017.pdf...
  ✓ Complete

[6/25] Rank_06_CITlab_ARGUS_for_historical_handwritten_2016.pdf...
  ✓ Complete

[7/25] Rank_07_HTR-VT_Handwritten_Text_Recognition_wit_2024.pdf...
  ✓ Complete

[8/25] Rank_08_End-to-End_Page-Level_Assessment_of_Hand_2023.

In [9]:
# ============================================================================
# DEBUG CELL: Check Dataset Contents
# ============================================================================
import os
from pathlib import Path

print("🔍 Checking your dataset folder...\n")

dataset_path = "/kaggle/input/top-ranked-papers-25"

# Check if folder exists
if os.path.exists(dataset_path):
    print(f"✅ Dataset folder exists: {dataset_path}\n")
    
    # List all files
    print("📁 Contents:")
    for item in Path(dataset_path).rglob("*"):
        print(f"  - {item}")
else:
    print(f"❌ Dataset folder NOT found: {dataset_path}")
    print("\n📂 Available input folders:")
    for folder in Path("/kaggle/input").iterdir():
        print(f"  - {folder}")

🔍 Checking your dataset folder...

✅ Dataset folder exists: /kaggle/input/top-ranked-papers-25

📁 Contents:
  - /kaggle/input/top-ranked-papers-25/Rank_18_Benchmarking_Chinese_Text_Recognition_D_2021.pdf
  - /kaggle/input/top-ranked-papers-25/Rank_19_Have_convolutions_already_made_recurrenc_2020.pdf
  - /kaggle/input/top-ranked-papers-25/Rank_05_HMM-based_Indic_Handwritten_Word_Recogni_2017.pdf
  - /kaggle/input/top-ranked-papers-25/Rank_20_Uncovering_the_Handwritten_Text_in_the_M_2023.pdf
  - /kaggle/input/top-ranked-papers-25/Rank_03_Offline_Handwritten_Recognition_of_Malay_2017.pdf
  - /kaggle/input/top-ranked-papers-25/Rank_16_ANN-based_Innovative_Segmentation_Method_2009.pdf
  - /kaggle/input/top-ranked-papers-25/Rank_13_Spatial_Context-based_Self-Supervised_Le_2024.pdf
  - /kaggle/input/top-ranked-papers-25/Rank_06_CITlab_ARGUS_for_historical_handwritten_2016.pdf
  - /kaggle/input/top-ranked-papers-25/Rank_21_MetaHTR_Towards_Writer-Adaptive_Handwri_2021.pdf
  - /kaggle/input/top-

In [12]:
# ============================================================================
# CELL 1: Install Dependencies
# ============================================================================
print("📦 Installing required packages...")
!pip install -q groq PyPDF2
print("✅ Packages installed!")

# ============================================================================
# CELL 2: Setup with Dual API Keys and Extract ZIP
# ============================================================================
import os
import json
import zipfile
from pathlib import Path
import PyPDF2
from groq import Groq
from datetime import datetime
import time

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# Setup dual API keys for rate limit management
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
GROQ_API_KEY1 = user_secrets.get_secret("GROQ_API_KEY1")

groq_client = Groq(api_key=GROQ_API_KEY)
groq_client1 = Groq(api_key=GROQ_API_KEY1)

# Track which client to use
current_client_index = 0
clients = [groq_client, groq_client1]

def get_groq_client():
    """Rotate between API keys to handle rate limits"""
    global current_client_index
    client = clients[current_client_index]
    current_client_index = (current_client_index + 1) % len(clients)
    return client

print("✅ Groq clients ready (2 API keys configured)")

# Extract ZIP file
print("\n📦 Extracting papers from ZIP...")
zip_path = "/kaggle/input/top-ranked-papers-25"
extract_to = "/kaggle/working/extracted_papers"
Path(extract_to).mkdir(parents=True, exist_ok=True)

zip_files = list(Path(zip_path).glob("*.zip"))
print(f"Found {len(zip_files)} ZIP file(s)")

for zip_file in zip_files:
    print(f"Extracting: {zip_file.name}")
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

pdf_files = list(Path(extract_to).glob("*.pdf"))
print(f"✅ Extracted {len(pdf_files)} PDF files")

# ============================================================================
# CELL 3: Analyze Papers with Enhanced Architecture Analysis
# ============================================================================

def analyze_papers(papers_folder: str):
    """Analyze papers with detailed architecture and Malayalam-specific recommendations"""
    
    print("\n" + "="*80)
    print("📚 ANALYZING YOUR 25 RANKED PAPERS")
    print("="*80 + "\n")
    
    pdf_files = sorted(list(Path(papers_folder).glob("*.pdf")))
    
    if not pdf_files:
        print(f"❌ No PDF files found in {papers_folder}")
        return
    
    print(f"✅ Found {len(pdf_files)} PDF files\n")
    
    summaries = []
    
    # ============================================================================
    # STEP 1: Generate Enhanced Summaries with Architecture Details
    # ============================================================================
    print("📝 GENERATING DETAILED SUMMARIES WITH ARCHITECTURE ANALYSIS...")
    print("-" * 80)
    
    for i, pdf_path in enumerate(pdf_files, 1):
        print(f"\n[{i}/{len(pdf_files)}] {pdf_path.name[:70]}...")
        
        try:
            # Extract text from PDF
            pdf_text = ""
            with open(pdf_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                num_pages = min(15, len(pdf_reader.pages))
                for page_num in range(num_pages):
                    try:
                        pdf_text += pdf_reader.pages[page_num].extract_text() + "\n"
                    except:
                        continue
            
            if not pdf_text.strip():
                print(f"  ⚠️  Could not extract text, skipping...")
                continue
            
            title = pdf_path.stem.replace('_', ' ')
            
            # Enhanced prompt with architecture focus
            prompt = f"""Analyze this Malayalam/Indic HTR research paper in detail:

Title: {title}

Paper Text (first 15 pages):
{pdf_text[:12000]}

Provide comprehensive structured analysis:

**1. MAIN CONTRIBUTION**
Describe the key innovation and novelty (3-4 sentences)

**2. ARCHITECTURE DETAILS**
- Architecture Type: (Traditional/CNN-based/RNN-LSTM/CNN-LSTM hybrid/Attention-based/Transformer/Vision Transformer/Hybrid)
- Specific Model Name: (e.g., ResNet-LSTM-CTC, CRNN, TrOCR, HTR-Net, etc.)
- Feature Extractor: Detail the CNN/ViT backbone used (VGG, ResNet, EfficientNet, DeiT, etc.)
- Sequence Modeling: Detail RNN/LSTM/GRU/Transformer layers and configurations
- Attention Mechanism: Describe if self-attention, cross-attention, or multi-head attention is used
- Decoder Type: CTC, Attention-based, Transformer decoder, Beam search parameters
- Input Preprocessing: Image normalization, augmentation, resizing strategies
- Loss Function: CTC loss, Cross-entropy, Custom losses
- Layer Architecture: Number of layers, hidden units, dropout rates

**3. DATASET & TRAINING**
- Dataset name, size, splits
- Character/word vocabulary size
- Data augmentation techniques used
- Training hyperparameters (learning rate, batch size, epochs, optimizer)
- Hardware used (GPUs, training time)

**4. RESULTS & METRICS**
- CER (Character Error Rate) / WER (Word Error Rate) / Accuracy
- Comparison with baseline methods (include specific numbers)
- Inference speed/efficiency metrics

**5. MALAYALAM-SPECIFIC TECHNIQUES**
List techniques specifically relevant to Malayalam script:
- How conjuncts/ligatures are handled
- Unicode normalization strategies
- Script-specific preprocessing
- Character segmentation approaches

**6. APPLICABLE TECHNIQUES FOR HTR**
List 7-10 concrete techniques from this paper that can be applied to any HTR project

**7. ARCHITECTURE STRENGTHS**
Why this architecture works well for the task (3-4 points)

**8. LIMITATIONS & GAPS**
Specific limitations of the architecture or approach

Be extremely technical and detailed about architecture components."""
            
            # Try with alternating API keys
            max_retries = 3
            for retry in range(max_retries):
                try:
                    client = get_groq_client()
                    response = client.chat.completions.create(
                        model="llama-3.3-70b-versatile",
                        messages=[{"role": "user", "content": prompt}],
                        max_tokens=2500,
                        temperature=0.3
                    )
                    
                    summary = {
                        'rank': i,
                        'title': title,
                        'filename': pdf_path.name,
                        'detailed_summary': response.choices[0].message.content,
                        'pdf_path': str(pdf_path)
                    }
                    
                    summaries.append(summary)
                    print(f"  ✓ Complete")
                    break
                    
                except Exception as e:
                    if "rate_limit" in str(e).lower() and retry < max_retries - 1:
                        print(f"  ⚠️  Rate limit, switching API key...")
                        time.sleep(3)
                        continue
                    else:
                        print(f"  ✗ Error: {str(e)[:100]}")
                        break
            
            time.sleep(2)
            
        except Exception as e:
            print(f"  ✗ Error: {str(e)[:100]}")
            continue
    
    print(f"\n✅ Generated {len(summaries)} summaries")
    
    if len(summaries) == 0:
        print("❌ No summaries generated. Cannot proceed with comparison.")
        return None
    
    # ============================================================================
    # STEP 2: Generate Comprehensive Comparative Analysis
    # ============================================================================
    print("\n🔬 GENERATING COMPREHENSIVE COMPARATIVE ANALYSIS...")
    print("-" * 80)
    
    summaries_text = ""
    for summary in summaries:
        summaries_text += f"\n{'='*80}\n"
        summaries_text += f"PAPER #{summary['rank']}: {summary['title']}\n"
        summaries_text += f"{summary['detailed_summary']}\n"
    
    prompt = f"""Analyze Malayalam HTR research across {len(summaries)} papers with deep architectural insights.

Papers:
{summaries_text[:45000]}

Create comprehensive analysis in detailed prose paragraphs (NO bullet points):

# MALAYALAM HTR RESEARCH: COMPREHENSIVE ANALYSIS

## 1. RESEARCH LANDSCAPE OVERVIEW
Write 4-5 paragraphs covering:
- Overall evolution of Malayalam HTR research
- Major paradigm shifts in approaches
- Current state-of-the-art
- Key challenges being addressed

## 2. ARCHITECTURE EVOLUTION & COMPARISON
Write 6-8 detailed paragraphs analyzing:
- Traditional feature-based methods (HOG, SIFT, etc.) and their limitations
- Early CNN approaches and architectures used
- RNN-LSTM integration and sequence modeling evolution
- Attention mechanism adoption (self-attention, cross-attention)
- Transformer-based approaches (ViT, TrOCR, custom transformers)
- Hybrid architectures combining multiple approaches
- Vision Transformers vs CNN-based architectures for Malayalam
- Cite specific papers by rank number with exact architecture details

## 3. DETAILED ARCHITECTURE PATTERNS
Write 5-6 paragraphs comparing:
- Feature extraction backbones (VGG, ResNet, EfficientNet, ViT, DeiT) - which works best?
- Sequence modeling layers (LSTM, GRU, BiLSTM, Transformer encoders)
- Decoder strategies (CTC vs Attention vs Transformer decoder vs Hybrid)
- Input/output layer designs
- How Malayalam script characteristics influence architecture choices
- Include specific layer configurations from papers

## 4. MALAYALAM SCRIPT-SPECIFIC CHALLENGES & SOLUTIONS
Write 4-5 paragraphs on:
- Conjunct/ligature handling approaches across papers
- Character segmentation strategies
- Unicode normalization techniques
- How different architectures handle Malayalam's complex script
- Best practices that emerged

## 5. PERFORMANCE ANALYSIS
Write 4-5 paragraphs covering:
- Best CER/WER/Accuracy metrics achieved (cite paper rank with exact numbers)
- Performance vs architecture complexity trade-offs
- Training efficiency and inference speed comparisons
- Which architectures give best accuracy for Malayalam
- Impact of dataset size on different architectures

## 6. DATASETS & TRAINING METHODOLOGIES
Write 4-5 paragraphs analyzing:
- Common datasets used (names, sizes, characteristics)
- Data augmentation strategies that work well
- Training hyperparameters and best practices
- Transfer learning and pre-training approaches
- Few-shot and meta-learning techniques

## 7. TOP 15 TECHNIQUES FOR MALAYALAM HTR
Write detailed paragraphs for each of the 15 most impactful techniques:
- For each technique: describe it, explain why it works, cite which papers (by rank) used it, and explain its impact on performance
- Focus on techniques proven effective for Malayalam specifically

## 8. ARCHITECTURE SELECTION GUIDE
Write 5-6 paragraphs providing:
- Comprehensive comparison of architecture families
- When to use CNN-LSTM vs Pure Transformer vs Hybrid
- Dataset size recommendations for each architecture type
- Computational resource requirements
- Practical trade-offs between accuracy and efficiency

## 9. GAPS & FUTURE RESEARCH DIRECTIONS
Write 4-5 paragraphs on:
- Unsolved problems in Malayalam HTR
- Architectural innovations needed
- Dataset and benchmark limitations
- Promising emerging techniques

## 10. RECOMMENDATIONS FOR YOUR MALAYALAM HTR PROJECT
Write 6-8 detailed paragraphs with actionable recommendations:

**ARCHITECTURE RECOMMENDATION:**
Based on all analyzed papers, recommend the BEST architecture for Malayalam HTR. Provide:
- Exact architecture to implement (e.g., "ResNet50 + BiLSTM + Attention + CTC")
- Detailed reasoning for each component choice
- Why this architecture is optimal for Malayalam script specifically
- Expected performance ranges
- Which papers (by rank) validate this recommendation

**IMPLEMENTATION ROADMAP:**
- Specific model layers and configurations to use
- Dataset preparation strategies
- Training approach and hyperparameters
- Data augmentation pipeline
- Evaluation metrics to track

**CRITICAL SUCCESS FACTORS:**
- Key techniques that MUST be implemented
- Common pitfalls to avoid (cite papers that encountered them)
- Malayalam-specific preprocessing requirements
- Performance optimization strategies

**ALTERNATIVE APPROACHES:**
- Secondary architecture recommendations if resources are limited
- Simpler baseline to start with
- Advanced variations for higher accuracy

IMPORTANT: 
- Write ONLY in flowing paragraphs, absolutely NO bullet points or numbered lists
- Always cite specific paper ranks when making claims
- Include exact metrics (CER/WER/Accuracy percentages) whenever mentioned
- Explain WHY techniques work, not just WHAT they are
- Be highly technical and specific about architecture details
- Focus on actionable insights for implementation"""
    
    max_retries = 3
    for retry in range(max_retries):
        try:
            print(f"  🤖 Groq analyzing patterns (attempt {retry + 1}/{max_retries})...")
            
            client = get_groq_client()
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=8000,
                temperature=0.3
            )
            
            comparison_report = response.choices[0].message.content
            print("✅ Comparative analysis complete")
            break
            
        except Exception as e:
            if "rate_limit" in str(e).lower() and retry < max_retries - 1:
                print(f"  ⚠️  Rate limit hit, switching to alternate API key...")
                time.sleep(5)
                continue
            else:
                print(f"⚠️  Error: {e}")
                comparison_report = f"Error generating comparison: {e}"
                break
    
    # ============================================================================
    # STEP 3: Export Enhanced Results
    # ============================================================================
    print("\n📄 EXPORTING ENHANCED RESULTS...")
    print("-" * 80)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_folder = "/kaggle/working/analysis_results"
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # Enhanced Markdown Report
    md_file = os.path.join(output_folder, f"malayalam_htr_analysis_enhanced_{timestamp}.md")
    with open(md_file, 'w', encoding='utf-8') as f:
        f.write("# MALAYALAM HTR RESEARCH: COMPREHENSIVE ARCHITECTURAL ANALYSIS\n\n")
        f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"**Papers Analyzed:** {len(summaries)}\n")
        f.write(f"**Analysis Focus:** Architecture, Malayalam-specific techniques, Implementation recommendations\n")
        f.write(f"**Powered by:** Groq (Llama 3.3-70B) with Dual API\n\n")
        
        f.write("---\n\n")
        f.write("## 📊 EXECUTIVE SUMMARY\n\n")
        f.write(f"This report analyzes {len(summaries)} research papers on Malayalam Handwritten Text Recognition, ")
        f.write("with deep focus on architectural patterns, Malayalam script-specific challenges, and ")
        f.write("actionable recommendations for building an effective Malayalam HTR system.\n\n")
        
        f.write("---\n\n")
        f.write("## 🔬 COMPREHENSIVE COMPARATIVE ANALYSIS\n\n")
        f.write(comparison_report)
        f.write("\n\n---\n\n")
        
        f.write("## 📑 PAPERS ANALYZED - QUICK REFERENCE\n\n")
        f.write("| Rank | Title | Key Architecture |\n")
        f.write("|------|-------|------------------|\n")
        for s in summaries:
            title_short = s['title'][:60] + "..." if len(s['title']) > 60 else s['title']
            # Extract architecture info from summary if possible
            arch_info = "See detailed summary"
            f.write(f"| #{s['rank']} | {title_short} | {arch_info} |\n")
        
        f.write("\n\n---\n\n")
        f.write("## 📚 DETAILED PAPER SUMMARIES\n\n")
        
        for s in summaries:
            f.write(f"\n### 📄 PAPER #{s['rank']}: {s['title']}\n\n")
            f.write(f"**File:** `{s['filename']}`\n\n")
            f.write("---\n\n")
            f.write(s['detailed_summary'])
            f.write("\n\n---\n\n")
    
    # Enhanced Text Report
    txt_file = md_file.replace('.md', '.txt')
    with open(txt_file, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("MALAYALAM HTR RESEARCH: COMPREHENSIVE ARCHITECTURAL ANALYSIS\n")
        f.write("="*80 + "\n")
        f.write(f"Papers Analyzed: {len(summaries)}\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Focus: Architecture + Malayalam-specific + Implementation\n")
        f.write("="*80 + "\n\n")
        
        f.write("COMPREHENSIVE COMPARATIVE ANALYSIS\n" + "-"*80 + "\n\n")
        f.write(comparison_report)
        f.write("\n\n" + "="*80 + "\n\n")
        
        f.write("DETAILED PAPER SUMMARIES\n" + "="*80 + "\n\n")
        for s in summaries:
            f.write(f"\nPAPER #{s['rank']}: {s['title']}\n")
            f.write("-"*80 + "\n")
            f.write(f"File: {s['filename']}\n\n")
            f.write(s['detailed_summary'])
            f.write("\n\n" + "="*80 + "\n\n")
    
    # Enhanced JSON Export
    json_file = os.path.join(output_folder, f"summaries_enhanced_{timestamp}.json")
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump({
            'metadata': {
                'total_papers': len(summaries),
                'timestamp': timestamp,
                'analysis_type': 'comprehensive_architectural',
                'focus_areas': [
                    'Architecture comparison',
                    'Malayalam-specific techniques',
                    'Implementation recommendations',
                    'Performance analysis'
                ]
            },
            'summaries': summaries,
            'comparative_analysis': comparison_report
        }, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Enhanced results exported to:")
    print(f"  📄 Markdown: {md_file}")
    print(f"  📄 Text: {txt_file}")
    print(f"  📄 JSON: {json_file}")
    
    print("\n" + "="*80)
    print("✅ COMPREHENSIVE ANALYSIS COMPLETE!")
    print("="*80)
    print(f"\n📊 SUMMARY:")
    print(f"  • Papers analyzed: {len(summaries)}")
    print(f"  • Architecture details: ✓ Included")
    print(f"  • Malayalam-specific techniques: ✓ Included")
    print(f"  • Implementation recommendations: ✓ Included")
    print(f"  • Output folder: {output_folder}")
    print("\n💡 Download files from Kaggle Output panel (right sidebar) ➡️")
    print("\n🎯 KEY DELIVERABLES:")
    print("  1. Detailed architecture comparison across papers")
    print("  2. Malayalam script-specific solutions")
    print("  3. Recommended architecture for your HTR project with reasoning")
    print("  4. Implementation roadmap and best practices")
    print("="*80 + "\n")
    
    return {
        'summaries': summaries,
        'comparison_report': comparison_report,
        'output_files': {
            'markdown': md_file,
            'text': txt_file,
            'json': json_file
        }
    }

# ============================================================================
# CELL 4: RUN ENHANCED ANALYSIS
# ============================================================================

print("\n🚀 Starting enhanced analysis with dual API keys...")
print("📋 Analysis includes:")
print("  • Detailed architecture breakdowns")
print("  • Malayalam script-specific techniques")
print("  • Performance comparisons")
print("  • Recommended architecture with reasoning")
print("  • Implementation roadmap\n")

result = analyze_papers("/kaggle/input/top-ranked-papers-25")

print("\n✨ Done! Check 'analysis_results' folder for comprehensive reports!")
print("📖 The report includes specific recommendations for YOUR Malayalam HTR project!")

📦 Installing required packages...
✅ Packages installed!
✅ Groq clients ready (2 API keys configured)

📦 Extracting papers from ZIP...
Found 0 ZIP file(s)
✅ Extracted 0 PDF files

🚀 Starting enhanced analysis with dual API keys...
📋 Analysis includes:
  • Detailed architecture breakdowns
  • Malayalam script-specific techniques
  • Performance comparisons
  • Recommended architecture with reasoning
  • Implementation roadmap


📚 ANALYZING YOUR 25 RANKED PAPERS

✅ Found 25 PDF files

📝 GENERATING DETAILED SUMMARIES WITH ARCHITECTURE ANALYSIS...
--------------------------------------------------------------------------------

[1/25] Rank_01_TMIXT_A_process_flow_for_Transcribing_M_2019.pdf...
  ✓ Complete

[2/25] Rank_02_Handwritten_Character_Recognition_In_Mal_2014.pdf...
  ✓ Complete

[3/25] Rank_03_Offline_Handwritten_Recognition_of_Malay_2017.pdf...
  ⚠️  Rate limit, switching API key...
  ✓ Complete

[4/25] Rank_04_Spatial_Domain_Feature_Extraction_Method_2021.pdf...
  ⚠️  Rate limit,

In [5]:
pip install PyPDF2 groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [8]:
# ============================================================================
# COMPLETE ANALYSIS: All 25 Malayalam/Indic HTR Papers
# ============================================================================
import json
import time
from pathlib import Path
import PyPDF2
from groq import Groq
from datetime import datetime
from kaggle_secrets import UserSecretsClient

print("="*80)
print("MALAYALAM HTR: ANALYZING ALL 25 PAPERS")
print("="*80)

# Setup API clients
try:
    user_secrets = UserSecretsClient()
    GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
    GROQ_API_KEY1 = user_secrets.get_secret("GROQ_API_KEY1")
    
    clients = [Groq(api_key=GROQ_API_KEY), Groq(api_key=GROQ_API_KEY1)]
    current_client = 0
    print("✅ API clients initialized successfully")
except Exception as e:
    print(f"❌ Error initializing API clients: {e}")
    exit(1)

def get_client():
    """Rotate between API clients"""
    global current_client
    client = clients[current_client]
    current_client = (current_client + 1) % len(clients)
    return client

def safe_api_call(prompt, max_tokens=2500, max_retries=3, delay=3):
    """Make API call with retry logic"""
    for retry in range(max_retries):
        try:
            client = get_client()
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=0.3
            )
            return response.choices[0].message.content
        except Exception as e:
            error_msg = str(e).lower()
            if 'rate' in error_msg or 'limit' in error_msg:
                wait_time = delay * (retry + 1) * 3
                print(f"    ⚠️  Rate limit. Waiting {wait_time}s...")
                time.sleep(wait_time)
            elif retry < max_retries - 1:
                print(f"    ⚠️  Retry {retry+1}/{max_retries}...")
                time.sleep(delay)
            else:
                print(f"    ❌ Failed: {str(e)[:100]}")
                return None
    return None

def extract_pdf_text(pdf_path, max_pages=15):
    """Extract text from PDF"""
    try:
        pdf_text = ""
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            num_pages = min(max_pages, len(pdf_reader.pages))
            
            for page_num in range(num_pages):
                try:
                    page_text = pdf_reader.pages[page_num].extract_text()
                    if page_text:
                        pdf_text += page_text + "\n"
                except:
                    continue
        return pdf_text.strip()
    except Exception as e:
        print(f"    ❌ PDF error: {str(e)[:50]}")
        return ""

# ============================================================================
# STEP 1: Find and Analyze All 25 Papers
# ============================================================================

papers_folder = Path("/kaggle/input/top-ranked-papers-25")
pdf_files = sorted(list(papers_folder.glob("Rank_*.pdf")))

print(f"\n📚 Found {len(pdf_files)} PDF files")
print("-" * 80)

summaries = []

for idx, pdf_path in enumerate(pdf_files, 1):
    # Extract rank number from filename
    filename = pdf_path.name
    try:
        rank = int(filename.split("Rank_")[1].split("_")[0])
    except:
        rank = idx
    
    title = filename.replace(f"Rank_{rank}_", "").replace(".pdf", "").replace("_", " ")
    
    print(f"\n[{idx}/{len(pdf_files)}] Paper #{rank}: {title[:60]}...")
    
    # Extract text
    pdf_text = extract_pdf_text(pdf_path)
    
    if not pdf_text:
        print(f"    ⚠️  Skipping - no text extracted")
        continue
    
    # Analyze with AI
    prompt = f"""Analyze this HTR research paper comprehensively:

Title: {title}

Paper Text (first 15 pages):
{pdf_text[:12000]}

Provide detailed structured analysis:

**1. MAIN CONTRIBUTION**
Key innovation and novelty (3-4 sentences)

**2. ARCHITECTURE DETAILS**
- Architecture Type: (CNN-based/RNN-LSTM/CNN-LSTM hybrid/Attention-based/Transformer/Vision Transformer/Hybrid)
- Specific Model: (e.g., ResNet-LSTM-CTC, CRNN, TrOCR, etc.)
- Feature Extractor: CNN/ViT backbone details
- Sequence Modeling: RNN/LSTM/GRU/Transformer layers
- Attention Mechanism: Type and implementation
- Decoder: CTC/Attention-based/Transformer/Beam search
- Input Processing: Normalization, augmentation, resizing
- Loss Function: Specific losses used
- Layer Details: Layers, hidden units, dropout

**3. DATASET & TRAINING**
- Dataset: Name, size, splits
- Vocabulary: Character/word count
- Augmentation: Techniques used
- Hyperparameters: LR, batch size, epochs, optimizer

**4. RESULTS & METRICS**
- Performance: CER/WER/Accuracy (exact numbers)
- Baseline Comparisons: Include specific metrics
- Key Findings: What worked best

**5. MALAYALAM/INDIC RELEVANCE**
How techniques apply to Malayalam script

**6. APPLICABLE TECHNIQUES**
List 7-10 concrete implementable techniques

**7. ARCHITECTURE STRENGTHS**
Why this approach works well

**8. LIMITATIONS**
Specific weaknesses and gaps

Be extremely technical and detailed."""
    
    print(f"    🤖 Analyzing...")
    analysis = safe_api_call(prompt, max_tokens=2500)
    
    if analysis:
        summaries.append({
            'rank': rank,
            'title': title,
            'filename': filename,
            'detailed_summary': analysis,
            'pdf_path': str(pdf_path)
        })
        print(f"    ✅ Complete")
        time.sleep(2)  # Rate limit protection
    else:
        print(f"    ❌ Failed to analyze")
    
    # Save progress every 5 papers
    if len(summaries) % 5 == 0:
        output_folder = Path("/kaggle/working/analysis_results")
        output_folder.mkdir(parents=True, exist_ok=True)
        
        progress_file = output_folder / f"progress_{len(summaries)}_papers.json"
        with open(progress_file, 'w', encoding='utf-8') as f:
            json.dump({'summaries': summaries}, f, indent=2, ensure_ascii=False)
        print(f"    💾 Progress saved: {len(summaries)} papers")

print(f"\n✅ ANALYSIS PHASE COMPLETE: {len(summaries)}/{len(pdf_files)} papers analyzed")

# ============================================================================
# STEP 2: Generate Comparative Analysis
# ============================================================================

print("\n" + "="*80)
print("🔬 GENERATING COMPARATIVE ANALYSIS")
print("="*80)

# Prepare summaries for context
summaries_text = ""
for s in summaries:
    summaries_text += f"\nPAPER #{s['rank']}: {s['title']}\n"
    summaries_text += f"{s['detailed_summary'][:800]}...\n"

sections = {}

# Section 1: Architecture Evolution
print("\n[1/5] Architecture Evolution & Overview...")
prompt = f"""Analyze {len(summaries)} Malayalam/Indic HTR papers.

Papers:
{summaries_text[:18000]}

Write 6-8 detailed paragraphs covering:
1. Research landscape evolution in Malayalam HTR
2. Architecture progression: Traditional → CNN → LSTM → Attention → Transformers
3. Key architectural patterns and innovations
4. What specifically works for Malayalam script complexity

Cite papers by rank number. Include metrics. Prose only, NO bullets."""

result = safe_api_call(prompt, max_tokens=2000)
sections['evolution'] = result if result else "Analysis incomplete."
print(f"  {'✓' if result else '✗'} Done")
if result: time.sleep(4)

# Section 2: Performance Analysis
print("\n[2/5] Performance Analysis...")
prompt = f"""Based on {len(summaries)} papers:

{summaries_text[:18000]}

Write 6-8 paragraphs:
1. Best CER/WER/Accuracy achieved (cite paper ranks + exact numbers)
2. Performance progression over research timeline
3. Top 10 most effective techniques for Malayalam HTR
4. Why each technique excels for Malayalam
5. Data augmentation strategies and their impact

Cite ranks. Include numbers. Prose only."""

result = safe_api_call(prompt, max_tokens=2000)
sections['performance'] = result if result else "Analysis incomplete."
print(f"  {'✓' if result else '✗'} Done")
if result: time.sleep(4)

# Section 3: Malayalam-Specific Solutions
print("\n[3/5] Malayalam Script Solutions...")
prompt = f"""From {len(summaries)} papers:

{summaries_text[:18000]}

Write 5-6 paragraphs:
1. Handling Malayalam conjuncts and ligatures
2. Character segmentation strategies
3. Unicode normalization techniques
4. Dataset characteristics and best practices
5. Why certain architectures suit Malayalam's complexity

Cite paper ranks. Technical depth. Prose only."""

result = safe_api_call(prompt, max_tokens=2000)
sections['malayalam_specific'] = result if result else "Analysis incomplete."
print(f"  {'✓' if result else '✗'} Done")
if result: time.sleep(4)

# Section 4: Architecture Selection Guide
print("\n[4/5] Architecture Selection Guide...")
prompt = f"""Based on {len(summaries)} papers:

{summaries_text[:18000]}

Write 5-6 paragraphs:
1. CNN-LSTM vs Transformer vs Hybrid architectures comparison
2. When to use each type (cite successful papers)
3. Dataset size requirements per architecture
4. Computational resources and training time
5. Accuracy vs efficiency trade-offs with examples

Cite ranks. Specifics. Prose only."""

result = safe_api_call(prompt, max_tokens=2000)
sections['selection_guide'] = result if result else "Analysis incomplete."
print(f"  {'✓' if result else '✗'} Done")
if result: time.sleep(4)

# Section 5: Implementation Recommendations
print("\n[5/5] 🎯 Implementation Recommendations...")
prompt = f"""Based on ALL {len(summaries)} papers, create actionable implementation guide:

{summaries_text[:18000]}

Write 7-9 detailed paragraphs:

**RECOMMENDED ARCHITECTURE:**
- EXACT architecture to implement (e.g., "ResNet50 + BiLSTM + Multi-head Attention + CTC")
- WHY each component is recommended
- Why OPTIMAL for Malayalam (conjuncts, ligatures, complexity)
- Expected CER/WER performance range
- Cite papers achieving best results with similar architectures

**IMPLEMENTATION ROADMAP:**
- Layer specifications and configurations
- Dataset preparation (size, format, splits)
- Training hyperparameters (LR, batch size, epochs, optimizer)
- Data augmentation pipeline
- Malayalam preprocessing requirements

**CRITICAL SUCCESS FACTORS:**
- 5-7 MUST-IMPLEMENT techniques
- Common pitfalls to avoid (cite papers)
- Essential Malayalam-specific preprocessing
- Performance optimization strategies

**ALTERNATIVES:**
- Limited resources: simpler baseline
- Maximum accuracy: advanced architecture
- Trade-offs between options

HIGHLY SPECIFIC and TECHNICAL. Implementation-ready guide. Cite ranks. Prose only."""

result = safe_api_call(prompt, max_tokens=2500)
sections['recommendations'] = result if result else "Analysis incomplete."
print(f"  {'✓' if result else '✗'} Done")

# ============================================================================
# STEP 3: Assemble Complete Report
# ============================================================================

print("\n" + "="*80)
print("📄 ASSEMBLING FINAL REPORT")
print("="*80)

comparison_report = f"""# MALAYALAM HTR RESEARCH: COMPREHENSIVE ANALYSIS

## 1. ARCHITECTURE EVOLUTION & RESEARCH LANDSCAPE

{sections.get('evolution', 'Analysis incomplete')}

## 2. PERFORMANCE ANALYSIS & EFFECTIVE TECHNIQUES

{sections.get('performance', 'Analysis incomplete')}

## 3. MALAYALAM SCRIPT-SPECIFIC SOLUTIONS

{sections.get('malayalam_specific', 'Analysis incomplete')}

## 4. ARCHITECTURE SELECTION GUIDE

{sections.get('selection_guide', 'Analysis incomplete')}

## 5. 🎯 IMPLEMENTATION RECOMMENDATIONS

{sections.get('recommendations', 'Analysis incomplete')}

---
*Analysis based on {len(summaries)} research papers*
"""

# Create output folder
output_folder = Path("/kaggle/working/analysis_results")
output_folder.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save Markdown Report
md_file = output_folder / f"COMPLETE_malayalam_htr_analysis_{timestamp}.md"
with open(md_file, 'w', encoding='utf-8') as f:
    f.write(f"# MALAYALAM HTR: COMPLETE ANALYSIS OF {len(summaries)} PAPERS\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"**Papers Analyzed:** {len(summaries)}/25\n\n")
    f.write("---\n\n")
    f.write("## 📊 EXECUTIVE SUMMARY\n\n")
    f.write(f"Comprehensive analysis of {len(summaries)} research papers on Malayalam/Indic ")
    f.write("Handwritten Text Recognition with architectural recommendations and implementation guide.\n\n")
    f.write("---\n\n")
    f.write(comparison_report)
    f.write("\n\n---\n\n")
    f.write("## 📚 DETAILED PAPER SUMMARIES\n\n")
    
    for s in sorted(summaries, key=lambda x: x['rank']):
        f.write(f"\n### PAPER #{s['rank']}: {s['title']}\n\n")
        f.write(f"**File:** `{s['filename']}`\n\n")
        f.write(s['detailed_summary'])
        f.write("\n\n---\n\n")

# Save Text Report
txt_file = md_file.with_suffix('.txt')
with open(txt_file, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write(f"MALAYALAM HTR: COMPLETE ANALYSIS OF {len(summaries)} PAPERS\n")
    f.write("="*80 + "\n\n")
    f.write(comparison_report)
    f.write("\n\n" + "="*80 + "\n")
    f.write("DETAILED PAPER SUMMARIES\n\n")
    for s in sorted(summaries, key=lambda x: x['rank']):
        f.write(f"\nPAPER #{s['rank']}: {s['title']}\n")
        f.write("-"*80 + "\n")
        f.write(s['detailed_summary'])
        f.write("\n\n")

# Save JSON
json_output = output_folder / f"COMPLETE_analysis_{timestamp}.json"
with open(json_output, 'w', encoding='utf-8') as f:
    json.dump({
        'metadata': {
            'total_papers_analyzed': len(summaries),
            'total_papers_available': len(pdf_files),
            'timestamp': timestamp,
            'sections_completed': sum(1 for s in sections.values() 
                                     if s and 'incomplete' not in s.lower())
        },
        'summaries': sorted(summaries, key=lambda x: x['rank']),
        'comparative_analysis': comparison_report,
        'sections': sections
    }, f, indent=2, ensure_ascii=False)

# Final Summary
print(f"\n✅ REPORTS SAVED:")
print(f"  📄 Markdown: {md_file.name}")
print(f"  📄 Text: {txt_file.name}")
print(f"  📄 JSON: {json_output.name}")

print("\n" + "="*80)
print("🎉 COMPLETE ANALYSIS FINISHED!")
print("="*80)

successful_sections = sum(1 for s in sections.values() 
                         if s and 'incomplete' not in s.lower())

print(f"\n📊 FINAL STATISTICS:")
print(f"  • Papers analyzed: {len(summaries)}/{len(pdf_files)}")
print(f"  • Analysis sections: {successful_sections}/5")
print(f"  • Output location: {output_folder}")

print(f"\n🎯 REPORT CONTENTS:")
sections_status = [
    ('Architecture Evolution', 'evolution'),
    ('Performance Analysis', 'performance'),
    ('Malayalam Solutions', 'malayalam_specific'),
    ('Selection Guide', 'selection_guide'),
    ('Implementation Plan', 'recommendations')
]

for name, key in sections_status:
    status = '✓' if sections.get(key) and 'incomplete' not in sections[key].lower() else '✗'
    print(f"  {status} {name}")

if successful_sections == 5:
    print(f"\n✨ ALL SECTIONS COMPLETED SUCCESSFULLY!")
    print(f"\n💡 Next Steps:")
    print(f"  1. Download the markdown report for easy reading")
    print(f"  2. Check the Implementation Recommendations section")
    print(f"  3. Review architecture details for your project")
else:
    print(f"\n⚠️  {5 - successful_sections} section(s) incomplete")
    print(f"   Re-run after 2-3 hours if rate limited")

print("="*80)

MALAYALAM HTR: ANALYZING ALL 25 PAPERS
✅ API clients initialized successfully

📚 Found 25 PDF files
--------------------------------------------------------------------------------

[1/25] Paper #1: Rank 01 TMIXT A process flow for Transcribing M 2019...
    🤖 Analyzing...
    ✅ Complete

[2/25] Paper #2: Rank 02 Handwritten Character Recognition In Mal 2014...
    🤖 Analyzing...
    ✅ Complete

[3/25] Paper #3: Rank 03 Offline Handwritten Recognition of Malay 2017...
    🤖 Analyzing...
    ✅ Complete

[4/25] Paper #4: Rank 04 Spatial Domain Feature Extraction Method 2021...
    🤖 Analyzing...
    ✅ Complete

[5/25] Paper #5: Rank 05 HMM-based Indic Handwritten Word Recogni 2017...
    🤖 Analyzing...
    ✅ Complete
    💾 Progress saved: 5 papers

[6/25] Paper #6: Rank 06 CITlab ARGUS for historical handwritten 2016...
    🤖 Analyzing...
    ✅ Complete

[7/25] Paper #7: Rank 07 HTR-VT Handwritten Text Recognition wit 2024...
    🤖 Analyzing...
    ✅ Complete

[8/25] Paper #8: Rank 08 En

In [9]:
# Print the complete analysis report
from pathlib import Path

# Path to your report
report_file = "/kaggle/working/analysis_results/COMPLETE_malayalam_htr_analysis_20251215_040550.txt"

print("="*80)
print("PRINTING COMPLETE MALAYALAM HTR ANALYSIS REPORT")
print("="*80)
print()

try:
    with open(report_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Print the entire content
    print(content)
    
    print()
    print("="*80)
    print("END OF REPORT")
    print("="*80)
    print(f"\n✅ Report printed successfully!")
    print(f"📊 Total characters: {len(content):,}")
    print(f"📄 Total lines: {content.count(chr(10)):,}")
    print("\n💡 You can now scroll up and copy-paste any section you need!")
    
except FileNotFoundError:
    print(f"❌ Error: File not found at {report_file}")
    print("\nAvailable files:")
    results_folder = Path("/kaggle/working/analysis_results")
    for file in sorted(results_folder.glob("*.txt")):
        print(f"  • {file.name}")
        
except Exception as e:
    print(f"❌ Error reading file: {e}")

PRINTING COMPLETE MALAYALAM HTR ANALYSIS REPORT

MALAYALAM HTR: COMPLETE ANALYSIS OF 25 PAPERS

# MALAYALAM HTR RESEARCH: COMPREHENSIVE ANALYSIS

## 1. ARCHITECTURE EVOLUTION & RESEARCH LANDSCAPE

The research landscape in Malayalam Handwritten Text Recognition (HTR) has undergone significant evolution over the years. Initially, traditional approaches focused on feature extraction and classification using statistical models, as seen in **PAPER #2**, which provided a comprehensive review of handwritten character recognition in Malayalam scripts. However, with the advent of deep learning, the field has shifted towards more advanced architectures, such as Convolutional Neural Networks (CNNs) and Recurrent Neural Networks (RNNs), as proposed in **PAPER #7**, which introduced a data-efficient Vision Transformer (ViT) approach for handwritten text recognition. This transition has led to improved performance and increased accuracy in recognizing Malayalam handwritten text.

The progression of